# MP32 — Satellite Image Explorer

[Brief](README.md) · [Validation](VALIDATION.md). Objective: read raster geometry, masks and spectral bands before interpreting an SR image. The measured example is licensed NAIP **aerial** training imagery; satellite-specific sensor calibration is not established by this lesson.

In [ ]:
from pathlib import Path
import json,sys
import numpy as np
import matplotlib.pyplot as plt
from affine import Affine
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/rssr').is_dir())
sys.path.insert(0,str(root/'stage_08_remote_sensing/MP32_satellite_image_explorer'))
from mp32_tools import pixel_to_map,masked_statistics
record=json.loads((root/'reports/mp32-raster-explorer.json').read_text())
assert record['source']['split']=='train'


## 1. Coordinates are not array indices

An image is C×H×W: bands, rows, columns. Its affine transform maps column c and row r into map coordinates: x=a(c+½)+b(r+½)+c₀ and y=d(c+½)+e(r+½)+f₀ for pixel **centers**. Pixel corners omit ½. The CRS defines units and coordinate meaning. A north-up projected raster usually has a>0, e<0, b=d=0; rotated grids require the full transform.

Example: 10-m pixels, upper-left corner (100,200), north-up. Pixel (row=2,column=3) has center (135,175), while its upper-left corner is (130,180). Changing the array dimensions without updating the transform changes its geographic extent.

In [ ]:
transform=Affine(10,0,100,0,-10,200)
assert pixel_to_map(transform,2,3)==(135,175)
assert pixel_to_map(transform,2,3,False)==(130,180)
print('center / corner:',pixel_to_map(transform,2,3),pixel_to_map(transform,2,3,False))


## 2. Nodata and band statistics

For band b, mean μ_b=(Σ valid x_b)/N_b. Validity is per band and excludes nonfinite values; an empty band returns unavailable, not zero. Nodata values are placeholders, not observations. A value of zero can be valid unless metadata/masks declare otherwise. A nodata mask is not a cloud mask.

Synthetic two-band 2×2 fixture: exclude the lower-left pixel (99). Remaining band-one values [1,3,5] average 3; band-two [2,4,6] average 4. Including 99 would bias the first mean to 27.

In [ ]:
fixture=np.array([[[1,3],[99,5]],[[2,4],[99,6]]])
mask=np.array([[True,True],[False,True]])
stats=masked_statistics(fixture,mask)
assert [s['mean'] for s in stats]==[3,4]
assert masked_statistics(fixture,np.zeros((2,2),bool))[0]['mean'] is None
print(stats)


## 3. Measured raster inspection

The experiment verifies the training scene SHA-256, reads only the 128×128 window at column/row (238,238), and records its CRS, transform, pixel size, band count, nodata and valid statistics. No held-out scene is read. Acquisition time identifies the sensor observation; the manifest retrieval time belongs to the cached source acquisition, not a fresh request today. Re-run deliberately with `python experiment.py` once the pinned local raster is available; notebook execution reads the measured report.

In [ ]:
print('Sensor:',record['sensor'])
print('Source:',record['source'])
print('Metadata:',record['metadata'])
print('License:',record['license'])
assert record['metadata']['bands']==4
assert all(s['valid_pixels']<=128*128 for s in record['metadata']['statistics'])


## 4. Band choice changes the display, not the underlying sensor

RGB assigns red/green/blue bands to screen R/G/B. The false-colour panel assigns NIR/red/green to screen R/G/B, so red-looking areas can indicate higher NIR, not red visible light. Both previews divide stored DN by 10000 and clip to the **same fixed [0,1] display range**; this is a display convention, not reflectance calibration. Transparent pixels are invalid. Histogram x-axis: DN/10000; y-axis: valid pixel counts; line labels identify the bands. No universal brightness or vegetation threshold is inferred.

In [ ]:
image=plt.imread(root/'reports/mp32-band-views.png')
plt.figure(figsize=(12,4));plt.imshow(image);plt.axis('off');plt.show()
for band,stat in zip(record['bands'],record['metadata']['statistics']):
    print(band, 'valid pixels',stat['valid_pixels'],'mean stored DN',stat['mean'])


## 5. Exercises — try before opening the solutions

1. Compute the center of pixel (0,0) in the 10-m fixture. What happens to the affine pixel sizes after ×2 upsampling that preserves extent?
2. Why is an all-invalid band not a zero-valued band? Can a nodata mask remove all clouds?
3. Can stronger red in the false-colour view prove vegetation health? Can this cached aerial example establish current satellite conditions?

<details><summary>Solutions</summary>

1. (105,195). Pixel sizes become +5 and −5 in the same projected CRS, with the corner origin unchanged. More samples do not prove new physical detail.
2. No observations exist, so the mean is unavailable. Valid zeros are observations. Clouds need suitable quality masks/criteria beyond nodata.
3. No: screen red represents NIR in this mapping, and health requires calibrated measurements and context. Acquisition is historical and the sensor is aerial; neither current conditions nor satellite generalization is demonstrated.
</details>

## Interpretation and connection to SR

The actual report above shows the valid counts and band means for this single crop. Differences between bands and display mappings demonstrate why a three-channel preview cannot describe all four-band information. They do not identify land-cover causes. Preserve transforms, CRS, masks and band order when constructing HR/LR pairs; evaluation must compare compatible, aligned valid pixels. Next: MP33 preprocessing. Remaining scientific limits: one crop, unverified physical calibration, no per-pixel cloud mask, no cross-sensor accuracy or learner-mastery claim.